In [ ]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

categories = ['comp.graphics', 'sci.space', 'talk.religion.misc']

newsgroups = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes')
)

data = []
labels = []

for i, category in enumerate(categories):
    indices = np.where(newsgroups.target == i)[0][:20]  # по 20 текстов
    for idx in indices:
        data.append(newsgroups.data[idx])
        labels.append(newsgroups.target_names[i])

print(len(data))

60


# CountVectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words='english')
count_matrix = vectorizer.fit_transform(data)

print(count_matrix.shape)

(60, 2342)


# Cosine similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def classify_text(input_text):
    # превращаем текст в вектор
    input_vec = vectorizer.transform([input_text])

    # считаем похожесть
    sim = cosine_similarity(input_vec, count_matrix)

    # находим самый похожий текст
    best_idx = np.argmax(sim)

    return labels[best_idx], sim[0][best_idx]

# Test 🧪

In [ ]:
test_sentences = [
    "The rocket launched into orbit.",
    "A new 3D rendering technique for graphics.",
    "Theological discussions on faith and god."
]

for s in test_sentences:
    cat, score = classify_text(s)
    print(f"Text: {s}")
    print(f"Category: {cat}")
    print(f"Similarity: {score:.4f}")
    print("------")

Text: The rocket launched into orbit.
Category: sci.space
Similarity: 0.0870
------
Text: A new 3D rendering technique for graphics.
Category: comp.graphics
Similarity: 0.1952
------
Text: Theological discussions on faith and god.
Category: talk.religion.misc
Similarity: 0.3430
------


# 🧠 Q1. 유사도가 0.0000이 나오는 이유

CountVectorizer는 학습 데이터에 등장한 단어들만을 기준으로 단어 사전을 생성한다.
따라서 입력 문장에 포함된 단어가 학습 데이터에 존재하지 않을 경우, 해당 단어들은 모두 무시된다.

이 경우 입력 문장은 모든 값이 0인 벡터로 변환되며, 기존 문서들과의 코사인 유사도를 계산할 때도 항상 0이 된다.
즉, 공통으로 등장하는 단어가 하나도 없기 때문에 유사도가 0으로 계산된다.

또한 CountVectorizer는 단어의 의미나 문맥을 고려하지 않고 단순히 단어의 등장 빈도만을 사용하기 때문에, 의미적으로 비슷한 문장이라도 단어가 다르면 유사도가 낮거나 0이 될 수 있다.


# 🚀 Q2. 성능 개선 방법 (예: TF-IDF 사용)

CountVectorizer 대신 TfidfVectorizer를 사용하면 성능이 향상될 수 있다.

TF-IDF는 단어의 단순 빈도가 아니라, 특정 문서에서 얼마나 중요한 단어인지를 반영한다.
즉, 여러 문서에 공통적으로 자주 등장하는 단어는 낮은 가중치를 가지며, 특정 문서에서만 자주 등장하는 단어는 높은 가중치를 갖는다.

이로 인해 중요한 키워드 중심으로 문서를 비교할 수 있게 되어, 코사인 유사도 계산의 정확도가 높아진다.
실험 결과, TF-IDF를 사용했을 때 더 높은 유사도 점수와 더 정확한 분류 결과를 얻을 수 있었다.
